# Task-Aware Joint-Action Double Q-Learning Scheduler

Double Q-learning for reliability-aware task assignment. The scheduler uses a
task-aware state and a masked joint action over processor core, voltage/frequency
level, and standby-replica count. Two cross-evaluated action-value tables reduce
maximization bias during temporal-difference updates.

## Imports and scheduling environment

In [ ]:
import os
from pathlib import Path
import time
import random
import csv
import sys
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def project_root() -> Path:
    """Find the repository root from a notebook or project subdirectory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found")


PROJECT_ROOT = project_root()
METHOD_DIR = PROJECT_ROOT / "notebooks" / "our_method"
if str(METHOD_DIR) not in sys.path:
    sys.path.insert(0, str(METHOD_DIR))

from scheduling_environment import Envir

## Scheduling-specific Double Q-learning agent

The tabular state key keeps the current task identity and scheduling progress, and
discretizes the continuous quantities that evolve during an episode:
each core's utilization and running system reliability. Actions use the environment's
masked joint encoding.

In [ ]:
class DoubleQLearningAgent:
    """Double Q-learning specialized for the task-aware CPS assignment environment."""

    def __init__(
        self,
        env: Envir,
        learning_rate=0.1,
        gamma=0.99,
        epsilon=1.0,
        epsilon_min=0.05,
        epsilon_decay=0.9995,
        utilization_bins=20,
        reliability_bins=20,
        seed=123,
    ):
        self.env = env
        self.learning_rate = float(learning_rate)
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)
        self.epsilon_min = float(epsilon_min)
        self.epsilon_decay = float(epsilon_decay)
        self.utilization_bins = int(utilization_bins)
        self.reliability_bins = int(reliability_bins)
        if self.utilization_bins < 2 or self.reliability_bins < 2:
            raise ValueError("The scheduling state needs at least two bins per continuous feature.")

        self.q1 = defaultdict(lambda: np.zeros(self.env.n_actions, dtype=np.float32))
        self.q2 = defaultdict(lambda: np.zeros(self.env.n_actions, dtype=np.float32))
        self.rng = np.random.default_rng(seed)

    def state_key(self, observation):
        """Encode the scheduling observation as a compact tabular key."""
        if self.env.current_task_idx is None:
            raise RuntimeError("A terminal observation has no scheduling decision state.")

        observation = np.asarray(observation, dtype=np.float32)
        if observation.shape != (self.env.obs_dim,):
            raise ValueError(
                f"Expected observation shape {(self.env.obs_dim,)}, got {observation.shape}."
            )

        # Core-load observations are normalized by capacity_limit and clipped to 1.5.
        normalized_loads = np.clip(observation[: self.env.cores] / 1.5, 0.0, 1.0)
        load_levels = np.minimum(
            (normalized_loads * self.utilization_bins).astype(np.int32),
            self.utilization_bins - 1,
        )

        running_reliability = float(np.clip(observation[-1], 0.0, 1.0))
        reliability_level = min(
            int(running_reliability * self.reliability_bins),
            self.reliability_bins - 1,
        )

        # The task id identifies the normalized task features loaded by Envir.
        return (
            int(self.env.current_task_idx),
            int(self.env.task_cursor),
            *load_levels.tolist(),
            int(reliability_level),
        )

    @staticmethod
    def valid_actions(action_mask):
        valid = np.flatnonzero(np.asarray(action_mask, dtype=np.bool_))
        if valid.size == 0:
            raise RuntimeError("The scheduling environment returned an empty action mask.")
        return valid

    def select_action(self, state, action_mask, explore=True, deterministic=False):
        valid = self.valid_actions(action_mask)
        if explore and self.rng.random() < self.epsilon:
            return int(self.rng.choice(valid))

        # Both estimators vote on the behavior action.
        values = self.q1[state][valid] + self.q2[state][valid]
        best = valid[np.flatnonzero(np.isclose(values, values.max()))]
        if deterministic:
            return int(best[0])
        return int(self.rng.choice(best))

    def update(self, state, action, reward, next_state, next_mask, done):
        # Select with the table being updated and evaluate with the other table.
        update_q1 = bool(self.rng.integers(0, 2))
        selected_table = self.q1 if update_q1 else self.q2
        evaluation_table = self.q2 if update_q1 else self.q1

        target = float(reward)
        if not done:
            valid_next = self.valid_actions(next_mask)
            selection_values = selected_table[next_state][valid_next]
            greedy_candidates = valid_next[
                np.flatnonzero(np.isclose(selection_values, selection_values.max()))
            ]
            greedy_action = int(self.rng.choice(greedy_candidates))
            target += self.gamma * float(evaluation_table[next_state][greedy_action])

        td_error = target - float(selected_table[state][action])
        selected_table[state][action] += self.learning_rate * td_error
        return float(td_error), ("q1" if update_q1 else "q2")

    def train(
        self,
        time_elapse,
        episode_elapse,
        csv_dir,
        tasks_n,
        episodes=15_000,
        save_dir="double_q_learning_results",
        policy_dir="double_q_learning_policy",
        log_last_k=100,
        seed=42,
        alphas=(0.3, 0.0, 0.55, 0.15),
    ):
        """Train the scheduler on one generated task set."""
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(policy_dir, exist_ok=True)
        random.seed(seed)
        np.random.seed(seed)
        self.rng = np.random.default_rng(seed)

        csv_files = []
        for task_number in range(1, int(tasks_n) + 1):
            candidates = (
                os.path.join(csv_dir, f"train_task{task_number}.csv"),
                os.path.join(csv_dir, f"task{task_number}.csv"),
            )
            task_file = next((path for path in candidates if os.path.exists(path)), None)
            if task_file is None:
                raise FileNotFoundError(
                    f"Task {task_number} was not found as train_task{task_number}.csv "
                    f"or task{task_number}.csv under {csv_dir}."
                )
            csv_files.append(task_file)

        self.env.alpha = tuple(alphas)
        self.env.read_data(csv_files)
        self.env.print_task_details()

        rewards_hist = []
        detail_log_file_path = os.path.join(save_dir, "detail_tasks_after_learn2.txt")
        open(detail_log_file_path, "w").close()

        for episode in range(1, int(episodes) + 1):
            episode_start = time.time()
            observation, _ = self.env.reset(
                seed=(seed if episode == 1 else None),
                shuffle_tasks=True,
            )
            state = self.state_key(observation)
            done = False
            episode_reward = 0.0
            step_rewards = []
            step_infos = []
            td_errors = []
            q1_updates = 0
            q2_updates = 0

            while not done:
                step_start = time.time()
                action_mask = self.env.joint_action_mask()
                action = self.select_action(state, action_mask, explore=True)
                next_observation, reward, terminated, truncated, info = self.env.step(action)
                done = bool(terminated or truncated)

                next_state = None if done else self.state_key(next_observation)
                next_mask = None if done else self.env.joint_action_mask()
                td_error, updated_table = self.update(
                    state,
                    action,
                    reward,
                    next_state,
                    next_mask,
                    done,
                )
                td_errors.append(abs(td_error))
                q1_updates += int(updated_table == "q1")
                q2_updates += int(updated_table == "q2")

                episode_reward += reward
                step_rewards.append(float(reward))
                step_infos.append(info)
                if not done:
                    state = next_state

                elapsed = time.time() - step_start
                worst_time = time_elapse[0]
                average_time, time_count = time_elapse[1]
                time_elapse[0] = max(worst_time, elapsed)
                time_elapse[1] = (
                    (average_time * time_count + elapsed) / (time_count + 1),
                    time_count + 1,
                )

            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            rewards_hist.append(episode_reward / max(1, len(step_rewards)))

            episode_elapsed = time.time() - episode_start
            worst_episode = episode_elapse[0]
            average_episode, episode_count = episode_elapse[1]
            episode_elapse[0] = max(worst_episode, episode_elapsed)
            episode_elapse[1] = (
                (average_episode * episode_count + episode_elapsed) / (episode_count + 1),
                episode_count + 1,
            )

            if episode % 100 == 0 or episode == 1 or episode == episodes:
                print(
                    f"[Double Q-learning] Episode {episode}/{episodes} "
                    f"| avg_step_reward={rewards_hist[-1]:.4f} "
                    f"| mean_abs_td={np.mean(td_errors):.4f} "
                    f"| epsilon={self.epsilon:.4f} "
                    f"| states={len(set(self.q1) | set(self.q2))} "
                    f"| updates=({q1_updates},{q2_updates})"
                )

            if episode > episodes - log_last_k:
                with open(detail_log_file_path, "a") as log_file:
                    for step_number, info in enumerate(step_infos):
                        loads = "[" + ",".join(
                            f"{utilization:.4f}" for utilization in info["projected_state"]
                        ) + "]"
                        log_file.write(
                            f"Episode {episode}, step {step_number}, "
                            f"task={info['task']} ({info['task_name']}), "
                            f"action={info['action_idx']}, core={info['core_select']}, "
                            f"vf={info['vf_select']}, rep={info['replica_n']}, "
                            f"placements={info['placements']}, "
                            f"Reward={step_rewards[step_number]:.6f}, "
                            f"ProjectedUtil={loads}, "
                            f"TaskRel={info['task_reliability']:.8f}, "
                            f"SystemRel={info['system_reliability']:.8f}, "
                            f"CapacityViolation={info['capacity_violation']}, "
                            f"ThermalViolation={info['thermal_violation']}\n"
                        )

        all_states = sorted(set(self.q1) | set(self.q2))
        states = np.asarray(all_states, dtype=np.int32)
        q1_values = np.stack([self.q1[state] for state in all_states])
        q2_values = np.stack([self.q2[state] for state in all_states])
        np.savez_compressed(
            os.path.join(policy_dir, "double_q_tables.npz"),
            states=states,
            q1_values=q1_values,
            q2_values=q2_values,
            utilization_bins=self.utilization_bins,
            reliability_bins=self.reliability_bins,
            n_actions=self.env.n_actions,
            task_feature_keys=np.asarray(self.env.TASK_FEATURE_KEYS),
        )

        if len(rewards_hist) >= 2:
            window_size = min(len(rewards_hist), max(5, int(episodes * 0.06)))
            kernel = np.ones(window_size) / window_size
            smoothed = np.convolve(rewards_hist, kernel, mode="valid")
            plt.figure(figsize=(12, 6))
            plt.plot(smoothed, label=f"Smoothed Rewards (window={window_size})")
            plt.xlabel("Episodes")
            plt.ylabel("Reward per step")
            plt.title("Task-aware joint-action Double Q-learning")
            plt.legend()
            plt.grid()
            plt.savefig(
                os.path.join(save_dir, "smoothed_rewards_Double_Q_learning.png"),
                dpi=300,
                bbox_inches="tight",
            )
            plt.close()

        return rewards_hist

    def evaluate(self, episodes=20, seed=1000, deterministic=True):
        """Evaluate the learned tables and return per-step and aggregate metrics."""
        rows = []
        for episode in range(int(episodes)):
            observation, _ = self.env.reset(seed=seed + episode, shuffle_tasks=True)
            state = self.state_key(observation)
            done = False

            while not done:
                action_mask = self.env.joint_action_mask()
                action = self.select_action(
                    state,
                    action_mask,
                    explore=False,
                    deterministic=deterministic,
                )
                observation, reward, terminated, truncated, info = self.env.step(action)
                done = bool(terminated or truncated)
                rows.append({
                    "episode": episode,
                    "reward": reward,
                    "task": info["task"],
                    "core": info["core_select"],
                    "vf": info["vf_select"],
                    "replica_n": info["replica_n"],
                    "replica_penalty": info["replica_penalty"],
                    "task_reliability": info["task_reliability"],
                    "system_reliability": info["system_reliability"],
                    "capacity_violation": info["capacity_violation"],
                    "thermal_violation": info["thermal_violation"],
                    "max_core_util": float(np.max(info["projected_state"])),
                })
                if not done:
                    state = self.state_key(observation)

        frame = pd.DataFrame(rows)
        summary = {
            "mean_reward": float(frame["reward"].mean()),
            "capacity_violation_rate": float(frame["capacity_violation"].mean()),
            "thermal_violation_rate": float(frame["thermal_violation"].mean()),
            "mean_task_reliability": float(frame["task_reliability"].mean()),
            "mean_replica_n": float(frame["replica_n"].mean()),
            "mean_replica_penalty": float(frame["replica_penalty"].mean()),
            "mean_max_core_util": float(frame["max_core_util"].mean()),
        }
        return summary, frame

## Self-test

In [ ]:
# Lightweight scheduler and learning-update self-test.
import tempfile

_tmp = tempfile.mkdtemp(prefix="double_q_learning_smoke_")
_synthetic = [
    (0.20, 1.0, 0.20, 2.0e-10, 2.0e-10, 0.020),
    (0.45, 1.2, 0.38, 3.0e-10, 3.0e-10, 0.030),
    (0.15, 0.6, 0.25, 1.0e-10, 1.0e-10, 0.010),
    (0.30, 0.9, 0.33, 2.5e-10, 2.5e-10, 0.025),
]

for task_number, (execution, deadline, utilization, peak, alpha_c, i0) in enumerate(
    _synthetic, start=1
):
    pd.DataFrame([{
        "#": execution,
        "static": 0.1,
        "avg_power": 1.0,
        "alpha_C": alpha_c,
        "I0_exp_term": i0,
        "util": utilization,
        "peak": peak,
        "deadline": deadline,
        "min": 0.0,
    }]).to_csv(os.path.join(_tmp, f"train_task{task_number}.csv"), index=False)

_test_env = Envir(
    initial_voltage=1.1,
    initial_frequency=2e9,
    cores=4,
    main_tasks=4,
    alpha=(0.3, 0.0, 0.55, 0.15),
    util_load=0.4,
)
_test_files = [
    os.path.join(_tmp, f"train_task{task_number}.csv")
    for task_number in range(1, 5)
]
_test_env.read_data(_test_files)

_observation, _ = _test_env.reset(seed=7, shuffle_tasks=False)
_other_task_observation = _test_env._observation_for(1)
assert _observation.shape == (_test_env.obs_dim,)
assert not np.allclose(_observation, _other_task_observation)

for _joint_action in [(0, 0, 0), (3, 4, 10), (2, 1, 4)]:
    assert _test_env.decode_action(_test_env.encode_action(*_joint_action)) == _joint_action

_action_replica_2 = _test_env.encode_action(0, 4, 2)
_action_replica_8 = _test_env.encode_action(0, 4, 8)
_projection_2 = _test_env.project_action(_action_replica_2, task_idx=0)
_projection_8 = _test_env.project_action(_action_replica_8, task_idx=0)
assert np.allclose(
    _projection_2["projected_state"],
    _projection_8["projected_state"],
), "Standby replicas must not multiply normal CPU utilization."

_action_mask = _test_env.joint_action_mask(task_idx=0)
assert _action_mask[_action_replica_2] == _action_mask[_action_replica_8]

_test_agent = DoubleQLearningAgent(
    _test_env,
    epsilon_decay=0.9,
    utilization_bins=8,
    reliability_bins=8,
    seed=11,
)
_time = [0, (0, 0)]
_episode_time = [0, (0, 0)]
_history = _test_agent.train(
    _time,
    _episode_time,
    csv_dir=_tmp,
    tasks_n=4,
    episodes=3,
    save_dir=os.path.join(_tmp, "logs"),
    policy_dir=os.path.join(_tmp, "policy"),
    log_last_k=1,
    seed=11,
)
_summary, _rows = _test_agent.evaluate(episodes=2, seed=1000, deterministic=True)

assert len(_history) == 3
assert len(_test_agent.q1) > 0
assert len(_test_agent.q2) > 0
assert _summary["capacity_violation_rate"] == 0.0
assert os.path.exists(os.path.join(_tmp, "policy", "double_q_tables.npz"))

print("Double Q-learning smoke test passed.")
print("Observation dimension:", _test_env.obs_dim)
print("Joint action count:", _test_env.n_actions)
print("Q1 states:", len(_test_agent.q1), "| Q2 states:", len(_test_agent.q2))
print("Evaluation summary:", _summary)

## Experiment runner

In [ ]:
def run_experiment_sweep(
    cores=4,
    episodes=None,
    data_root=None,
    output_root=None,
    seed=123,
):
    """Train and evaluate Double Q-learning over ten utilization load points."""
    data_root = Path(data_root or PROJECT_ROOT / "data" / "generated")
    output_root = Path(output_root or PROJECT_ROOT / "artifacts" / "double_q_learning")
    if episodes is None:
        episodes = int(15_000 * (cores / 4))

    tasks_n = int(cores * 10)
    all_summaries = []

    for load_point in range(1, 11):
        csv_dir = os.path.join(
            data_root,
            f"tsp-{cores}c",
            f"limit_{(load_point * cores) / 10}",
            "1",
        )
        csv_files = [
            os.path.join(csv_dir, f"task{task_number}.csv")
            for task_number in range(1, tasks_n + 1)
        ]
        missing = [path for path in csv_files if not os.path.exists(path)]
        if missing:
            print(
                f"[skip] load point {load_point}: missing {len(missing)} "
                f"task CSV(s) under {csv_dir}"
            )
            continue

        print(
            f"\n===== load point {load_point}/10 "
            f"| util_load={load_point * 0.1:.1f} ====="
        )

        env = Envir(
            initial_voltage=1.1,
            initial_frequency=2e9,
            cores=cores,
            main_tasks=tasks_n,
            alpha=(0.3, 0.0, 0.55, 0.15),
            util_load=load_point * 0.1,
            capacity_limit=1.1,
            tsp=1.5,
            rep_max=10,
        )
        agent = DoubleQLearningAgent(
            env,
            learning_rate=0.1,
            gamma=0.99,
            epsilon=1.0,
            epsilon_min=0.05,
            epsilon_decay=0.9995,
            utilization_bins=20,
            reliability_bins=20,
            seed=seed,
        )

        save_dir = os.path.join(output_root, "logs", str(load_point))
        policy_dir = os.path.join(output_root, "policy", str(load_point))
        info_dir = os.path.join(output_root, "info", str(load_point))
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(policy_dir, exist_ok=True)
        os.makedirs(info_dir, exist_ok=True)

        time_elapse = [0, (0, 0)]
        episode_elapse = [0, (0, 0)]
        rewards = agent.train(
            time_elapse,
            episode_elapse,
            csv_dir=csv_dir,
            tasks_n=tasks_n,
            episodes=episodes,
            save_dir=save_dir,
            policy_dir=policy_dir,
            log_last_k=100,
            seed=seed,
            alphas=(0.3, 0.0, 0.55, 0.15),
        )

        eval_summary, eval_rows = agent.evaluate(
            episodes=20,
            seed=seed + 10_000,
            deterministic=True,
        )
        eval_rows.to_csv(os.path.join(info_dir, "evaluation_rows.csv"), index=False)

        timing_path = os.path.join(info_dir, f"info{load_point}.csv")
        with open(timing_path, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["Episode Worst", "Episode Avg", "Task Worst", "Task Avg"])
            writer.writerow([
                round(episode_elapse[0], 5),
                round(episode_elapse[1][0], 5),
                round(time_elapse[0], 5),
                round(time_elapse[1][0], 5),
            ])

        row = {
            "load_point": load_point,
            "util_load": load_point * 0.1,
            "train_final_reward": rewards[-1],
            "learned_states": len(set(agent.q1) | set(agent.q2)),
            **eval_summary,
        }
        all_summaries.append(row)
        pd.DataFrame([row]).to_csv(
            os.path.join(info_dir, "evaluation_summary.csv"),
            index=False,
        )
        print("Evaluation:", row)

    summary_frame = pd.DataFrame(all_summaries)
    if not summary_frame.empty:
        summary_frame.to_csv(
            os.path.join(output_root, "double_q_learning_all_loads_summary.csv"),
            index=False,
        )
    return summary_frame


# Enable after running generate_task_sets.ipynb.
RUN_TRAINING = False

if RUN_TRAINING:
    sweep_summary = run_experiment_sweep(cores=4)
    display(sweep_summary)

## Evaluation configuration

Experiments use fixed task CSVs, load points, reward weights, episode counts, training
seeds, shuffled task order, deterministic evaluation seeds, action masks, and evaluation
metrics. Double Q-learning parameters include the learning rate, exploration schedule,
and state-bin counts.

Replica semantics: for the same task, core, and VF level, changing the
standby-replica count does not change normal utilization. Its cost remains in the
reliability term's load-dependent replica-pressure penalty.